In [ ]:
import os, json, subprocess, sys, time, glob, shutil, urllib.request
CATEGORIES = 'bugfix,feature'
print('python', sys.version.split()[0], '| categories:', CATEGORIES)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:800])


In [ ]:
os.chdir('/kaggle/working')
r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/mattdani21/ModelSwapper.git'], capture_output=True, text=True)
print(r.stdout[-300:], r.stderr[-300:])
os.chdir('/kaggle/working/ModelSwapper')
print(subprocess.run(['git', 'log', '-1', '--format=%h %ci'], capture_output=True, text=True).stdout)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pytest'], capture_output=True, text=True)
print('pytest install rc:', r.returncode)


In [ ]:
# Model pre-flight BEFORE the 19-min build. Input mount is flaky (runs 5-7:
# same dataset mounted at 13:30, missing later) and 23.7GB of models cannot
# coexist with the build on the ~20GB working quota. v9: right-sized set
# (14B Q4 + 8B Q4 = 14GB) so the HF fallback ALWAYS fits; input-first when
# the dataset happens to mount (speed), HF otherwise (reliability).
INPUT = '/kaggle/input/swapos-ggufs'
print(subprocess.run(['bash', '-c', 'ls -la /kaggle/input/ 2>&1; ls -la ' + INPUT + ' 2>&1 | head -6'], capture_output=True, text=True).stdout)
print(subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True).stdout)
HF = {
    'Qwen3-14B-Q4_K_M.gguf': 'https://huggingface.co/Qwen/Qwen3-14B-GGUF/resolve/main/Qwen3-14B-Q4_K_M.gguf',
    'Qwen3-8B-Q4_K_M.gguf': 'https://huggingface.co/Qwen/Qwen3-8B-GGUF/resolve/main/Qwen3-8B-Q4_K_M.gguf',
}
ROLES = ['reason', 'code', 'review']
FILES = {'reason': 'Qwen3-14B-Q4_K_M.gguf', 'code': 'Qwen3-8B-Q4_K_M.gguf', 'review': 'Qwen3-14B-Q4_K_M.gguf'}
os.makedirs('/kaggle/working/models', exist_ok=True)
MODEL_PATHS = {}
for role in ROLES:
    fname = FILES[role]
    pin = INPUT + '/' + fname
    if os.path.exists(pin) and os.path.getsize(pin) > 1e9:
        MODEL_PATHS[role] = pin
        print('input ok:', fname)
    else:
        dst = '/kaggle/working/models/' + fname
        if not (os.path.exists(dst) and os.path.getsize(dst) > 1e9):
            print('input MISSING', fname, '-> HF fallback download')
            subprocess.run(['wget', '-q', '-O', dst, HF[fname]], check=True, timeout=2400)
        MODEL_PATHS[role] = dst
        print('fallback:', fname, round(os.path.getsize(dst) / 1e9, 2), 'GB')
for p in MODEL_PATHS.values():
    assert os.path.exists(p) and os.path.getsize(p) > 1e9, 'model unavailable: ' + p
print(MODEL_PATHS)


In [ ]:
# CUDA layout on Kaggle (diag-verified): toolkit 12.8, driver lib ONLY in
# /usr/local/cuda-12.8/compat — symlink into toolkit lib64 for CMake.
COMPAT = '/usr/local/cuda-12.8/compat'
for lib in ['libcuda.so', 'libcuda.so.1']:
    src = COMPAT + '/' + lib
    if os.path.exists(src):
        subprocess.run(['bash', '-c', 'ln -sf ' + src + ' /usr/local/cuda/lib64/' + lib], capture_output=True, text=True)
        print('symlinked', src)
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/llama.cpp/build/bin/llama-server'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git'], capture_output=True, text=True, check=True)
    r = subprocess.run(['cmake', '-B', '/kaggle/working/llama.cpp/build', '-S', '/kaggle/working/llama.cpp',
                        '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release',
                        '-DCUDAToolkit_ROOT=/usr/local/cuda-12.8',
                        '-DCMAKE_LIBRARY_PATH=' + COMPAT + ':/usr/local/cuda/lib64'],
                       capture_output=True, text=True, timeout=900)
    print('cmake configure rc:', r.returncode)
    if r.returncode != 0:
        print(r.stdout[-1500:])
        print(r.stderr[-1500:])
    nproc = subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip() or '4'
    r = subprocess.run(['cmake', '--build', '/kaggle/working/llama.cpp/build', '--config', 'Release',
                        '-j', nproc, '--target', 'llama-server'], capture_output=True, text=True, timeout=3600)
    print('cmake build rc:', r.returncode)
    if r.returncode != 0:
        print(r.stdout[-1500:])
        print(r.stderr[-1500:])
bin_path = '/kaggle/working/llama.cpp/build/bin/llama-server'
assert os.path.exists(bin_path), 'llama-server build failed'
subprocess.run(['cp', bin_path, '/kaggle/working/llama-server'], check=True)
subprocess.run(['rm', '-rf', '/kaggle/working/llama.cpp'])
os.environ['LLAMA_SERVER'] = '/kaggle/working/llama-server'
os.environ['PATH'] = '/kaggle/working:' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = COMPAT + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print('llama-server ready')


In [ ]:
os.chdir('/kaggle/working/ModelSwapper')
env = dict(os.environ)
env['LLAMA_CONTEXT'] = '4096'
env['LLAMA_NGPU'] = '99'
cmd = [sys.executable, 'pipeline/run_pipeline.py',
       '--models-json', json.dumps(MODEL_PATHS),
       '--out', '/kaggle/working/pipeline-results.json',
       '--capsule-dir', '/kaggle/working/capsules',
       '--categories', CATEGORIES,
       '--port-base', '8950',
       '--max-iterations', '3',
       '--max-tokens', '2048']
print('running pipeline eval...')
t0 = time.time()
try:
    r = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=8 * 3600)
    print('pipeline rc:', r.returncode, '| wall:', round((time.time() - t0) / 60, 1), 'min')
    print((r.stdout or '')[-2500:])
    print((r.stderr or '')[-1000:])
except subprocess.TimeoutExpired:
    print('PIPELINE TIMEOUT after', round((time.time() - t0) / 60, 1), 'min — partial results kept')
# Keep the kernel output small: the models/ dir (fallback downloads) would
# bloat /kaggle/working into the output snapshot and break result fetches.
shutil.rmtree('/kaggle/working/models', ignore_errors=True)
print('models cleaned from working')


In [ ]:
os.chdir('/kaggle/working')
res_path = '/kaggle/working/pipeline-results.json'
if os.path.exists(res_path):
    d = json.load(open(res_path))
    print('PASS RATE:', d.get('pass_rate'), '|', d.get('tasks_passed'), '/', d.get('tasks_total'))
    print('per_category:', d.get('per_category'))
    print('mean_wall_clock_s:', d.get('mean_wall_clock_s'))
    print('mean_load_s:', d.get('mean_load_s'), '| mean_evict_s:', d.get('mean_evict_s'))
    with open('/kaggle/working/pipeline-summary.txt', 'w') as f:
        f.write(json.dumps({k: v for k, v in d.items() if k != 'results'}, indent=2))
else:
    print('NO RESULTS FILE')
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working', 'pipeline-results.json')
shutil.make_archive('/kaggle/working/capsules', 'zip', '/kaggle/working/capsules')
print('outputs:', sorted(glob.glob('/kaggle/working/results*') + glob.glob('/kaggle/working/capsules*')))
